## Retrieval-Augmented Generation

In this exercise, you'll put together a RAG system and compare outputs from RAG vs. just querying an LLM.

In [8]:
import os
import json

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

from openai import OpenAI

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


from sklearn.preprocessing import normalize

#pip install -U sentence-transformers

For this exercise, you'll be asking about Subspace-Constrained LoRA (SC-LoRA), a new technique described in [a recent article publised on arXiv.org](https://arxiv.org/abs/2505.23724). You've been provided the text of this article in the file 2505.23724v1.txt.

First, you'll need to set up a way to interact with the generator model. You can use the OpenAI class from the openai library for this. See [this page](https://developers.openai.com/api/docs/guides/text?lang=python) for more information. When you do this, you'll need to set the base_url to "https://openrouter.ai/api/v1" and to pass in your api key. Set the model to "openrouter/owl-alpha".

First, ask the model "How does SC-LoRA differ from regular LoRA?" without providing any additional context. Read through a few different responses. **Question:** Are the responses accurate, or does it seem like the model is just making up something that sounds plausible?

In [46]:
###load api key
with open('keys.json', 'r') as fi:
    credentials = json.load(fi)
api_key = credentials['api_key']

In [47]:
from openai import OpenAI
client = OpenAI(base_url="https://openrouter.ai/api/v1", 
                api_key = api_key )

query = "How does SC-LoRA differ from regular LoRA?"

response = client.responses.create(
    model="openrouter/owl-alpha",
    input=[{"role":"user", "content":query}]
)

print(response.output_text)

SC-LoRA differs from regular LoRA by incorporating a **sparse constraint** on the low-rank adaptation matrices. While standard LoRA introduces trainable low-rank matrices to adapt a pre-trained model, SC-LoRA adds a sparsity-inducing penalty (such as L1 regularization) to these matrices during training. This encourages many of the adaptation weights to become exactly zero, resulting in a **sparse and low-rank** representation. The key benefits of this approach include:

1.  **Reduced Parameter Count:** By promoting sparsity, SC-LoRA can achieve effective adaptation with even fewer non-zero parameters than regular LoRA, leading to more compact models.
2.  **Improved Interpretability:** Sparse matrices can make it easier to identify which specific weights or features are most important for the adaptation task.
3.  **Potential for Better Generalization:** Sparsity can act as a form of regularization, potentially preventing overfitting and improving the model's ability to generalize to uns

### Part 1: Manual RAG

In order to get more reliable responses, let's set up a RAG system.

In this first part, you'll build all of the pieces of the RAG system individually.

First, you'll need the retriever portion. Create a FAISS index to hold the text of the article. Encode this text using the all-MiniLM-L6-v2 encoder. Note that you'll want to divide the text into smaller chunks rather than encoding the whole article all at once. You could try, for example, the [RecursiveCharacterTextSplitter class from LangChain](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter). You'll need to specify a chunk_size and chunk_overlap. You could try a chunk_size of 500 and overlap of 50 as a starting point.

In [48]:
with open("data/2505.23724v1.txt", "r", encoding="utf-8") as f:
    text = f.read()

### using Recursive Character Text Splitter method
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,          
    chunk_overlap=50   
)
#apply the text to the text chunk splitter
chunks = text_splitter.split_text(text.replace('\n', ' '))

#add it to model
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ValueError: SentenceTransformer.encode() has been called with additional keyword arguments that this model does not use: ['normalize']. As per SentenceTransformer.get_model_kwargs(), this model does not accept any additional keyword arguments.

### FAISS index from similarity exercise

In [31]:
# abstracts_vectorized_normalized = normalize(lines)
query_embedding_normalized = normalize(embeddings)
query_embedding_normalized.shape

(112, 384)

In [32]:
#creating the index flat IP object
dimension = query_embedding_normalized.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(query_embedding_normalized)

In [33]:
D, I = index.search(query_embedding_normalized, k=5)
most_similar_articles = I[0]
most_similar_articles

In [40]:
save_list=[]
for x in most_similar_articles:
    check_line= chunks[x]
    save_list.append(check_line)

text_context = '\n'.join(save_list)
len(text_context)

2357

In [49]:
# [[chunks][i] for i in I[0]]

IndexError: list index out of range

Next, use the following as a system prompt:

```
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise. "
    f"Context: {context}"
)
```

Use the FAISS index to pull in relevant context to fill in the context. Try passing in this additional system prompt. Hint: you can do this by using the following messages in the client.chat.completions.create function

```
    messages=[
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": query,
        }
    ]
```

In [51]:
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise. "
    f"Context: {text_context}"
)

messages=[
    {
        "role": "system", #the instructions provided to the model/system
        "content": system_prompt,
    },
    {
        "role": "user", #The que3stion that the user inputs
        "content": "How does SC-LoRA differ from regular LoRA?",
    }
]

response = client.responses.create(
    model="openrouter/owl-alpha", 
    reasoning={"effort": "low"},
    input=messages
)

print(response.output_text)

Based on the provided context, SC-LoRA differs from regular LoRA by introducing a **subspace-constrained initialization framework** designed to balance efficient fine-tuning with knowledge preservation. While regular LoRA focuses on efficient adaptation, SC-LoRA specifically constrains the output of trainable adapters in a low-rank subspace to strengthen target knowledge while minimizing the impact on pre-trained knowledge. This approach addresses the trade-off between enhancing fine-tuning performance and preventing the forgetting of existing information, a challenge that previous methods could not handle simultaneously.


**response**: Based on the provided context, SC-LoRA differs from regular LoRA by introducing a **subspace-constrained initialization framework** designed to balance efficient fine-tuning with knowledge preservation. While regular LoRA focuses on efficient adaptation, SC-LoRA specifically constrains the output of trainable adapters in a low-rank subspace to strengthen target knowledge while minimizing the impact on pre-trained knowledge. This approach addresses the trade-off between enhancing fine-tuning performance and preventing the forgetting of existing information, a challenge that previous methods could not handle simultaneously.

How does adding this context change the results?

**previous response**: 
One of the key differences between SC-LoRA and regular LoRA lies in the way they handle the adaptation process. While regular LoRA introduces a low-rank decomposition to approximate weight updates, SC-LoRA incorporates a scaling mechanism that dynamically adjusts the magnitude of the updates. This allows for more precise control over the adaptation process, potentially leading to better performance in certain tasks.

-----

### Part 2: LangChain

You can also use the [LangChain library](https://www.langchain.com/) to help build your RAG system.

For the retriever, you can use the [HugginFaceEmbeddings class](https://reference.langchain.com/python/langchain-huggingface/embeddings/huggingface/HuggingFaceEmbeddings), using the all-MiniLM-L6-v2 model, to create your embedding model. There is also a [FAISS class](https://reference.langchain.com/python/langchain-community/vectorstores/faiss/FAISS), which has a useful from_texts method. Once you've created your vector store, use the [as_retriever method](https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html#langchain_community.vectorstores.faiss.FAISS.as_retriever) on it and save it to a variable named `retriever`.

In [54]:
from langchain_huggingface import HuggingFaceEmbeddings

### create embedding model ###
model_name = "sentence-transformers/all-mpnet-base-v2"
embedder = HuggingFaceEmbeddings(
    model_name=model_name
)

#Apply imbedding model to the previously processed text
## created a index with FAISS ##help from abigail 
vector_store = FAISS.from_texts(chunks, embedder)

#use the as_retriever method
# Fetch more documents for the MMR algorithm to consider
# But only return the top 5
retriever = vector_store.as_retriever(search_type="mmr", 
                               search_kwargs={"k": 5, "fetch_k": 50})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [56]:
retriever.invoke(query)

[Document(id='5b0ba2c2-a0e7-41a8-87d9-e256c2288b70', metadata={}, page_content='2021). Safety evaluation follows the setting in the previous section 4.1. For better comparabil-ity, we tune the learning rate of LoRA to 2e-5, 5e-5 and 1e-4. The learning rate for other methods is fixed to 2e-5. From the results in Table 2, we can observe that the data points exhibit a wider spread among these methods, both in utility and safety metric. Com- pared to the original model, SC-LoRA ( β= 0.9) exhibits almost no safety degradation, and achieves best utility, even surpassing full'),
 Document(id='effe3014-a49c-4286-b7ba-e1e9a113134f', metadata={}, page_content='matrices frozen. To express in mathematical form, LoRA adds low-rank adapters A, B to original weight matrix W0byW′=W0+BA, where W′, W0∈Rdout×din,A∈Rr×din,B∈Rdout×r, r≪min(din, dout). When fine-tuning, W0is kept frozen, and A, B are trainable parameters. From the default initialization scheme of LoRA, Ais initialized by Kaiming Initializat

For the generator, you can use the [ChatOpenAI class](https://python.langchain.com/docs/integrations/chat/openai/). Be sure to set base_url="https://openrouter.ai/api/v1", model_name="openrouter/owl-alpha", and openai_api_key= Your API key. Save this to a variable named `llm`.


In [57]:
llm = ChatOpenAI(
    model_name="openrouter/owl-alpha",
    base_url="https://openrouter.ai/api/v1",
    openai_api_key= api_key) 

Now that the two components have been created, we can combine them into a chat template using the [ChatPromptTemplate class](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html). We can set up a system prompt and the pass that in, like

In [59]:

system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

Now we need to connect all of the pieces together. Newer versions of LangChain commonly use LCEL (LangChain Expression Language)[https://www.langchain.com/blog/langchain-expression-language] to build pipelines where components are connected together using the | operator.

You'll need:

* A helper function that combines retrieved documents into a single string of context
* A pipeline that:
    - extracts the question from the input
    - sends the question to the retriever
    - formats the retrieved documents into context
    - passes both the context and question into the prompt
    - sends the completed prompt to the LLM

A simplified diagram looks like this:

input
   ↓
extract question
   ↓
retrieve documents
   ↓
format context
   ↓
prompt
   ↓
LLM

You can create this chain by using 

In [60]:

def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

rag_chain = (
    {
        "context": (
            RunnableLambda(lambda x: x["question"])
            | retriever
            | RunnableLambda(format_docs)
        ),
        "question": RunnableLambda(
            lambda x: x["question"]
        )
    }
    | prompt
    | llm
)


Take a minute to study this and see if you can figure out how the syntax works.

Finally, invoke your chain using:

In [61]:
response = rag_chain.invoke(
    {"question": query}
)

print(response.content)

SC-LoRA differs from regular LoRA primarily in its initialization strategy. While regular LoRA initializes the down-projection matrix **A** with Gaussian noise and the up-projection matrix **B** with zeros, SC-LoRA initializes **A** as $Q_r^\top W_0$ and **B** as $Q_r$. Here, $Q_r$ consists of $r$ orthonormal vectors derived from the eigenvectors of a specific covariance difference matrix ($\Delta Cov$). This specialized initialization is designed to better preserve the model's original knowledge and safety during fine-tuning.


```

Compare the output from this section with both previous approaches:
* LLM without retrieval
* Manual RAG
* LangChain RAG

The quality of the answers from (2) and (3) should be similar. The purpose of LangChain is not to improve the model itself. Rather, it provides abstractions that simplify the retrieval and generation workflow.